# FAILED - NOT USED

### CNN v4 - non overlap test

v3 but non-overlapping

v3 did 0.82

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 1738
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CFG = dict(region_s=896, win_s=8, stride_s=8, kernels=(15, 15, 15), filters=(16, 32, 64), pool=2, epochs=100, patience=10, batch=64, lr=1e-3, weight_decay=1e-4, lr_factor=0.5, lr_patience=2, min_lr=1e-5, select='val RMSE')
NAME = 'v4_k15_full'

In [2]:
DATA = Path('eeg_starter/data')
FS, C = 64, 20
WIN, STRIDE, N_SAMPLES = CFG['win_s'] * FS, CFG['stride_s'] * FS, CFG['region_s'] * FS

def load_split(name):
    seg = torch.load(DATA / f'segments_h1k_{name}.pt')
    S = torch.empty(len(seg), C, N_SAMPLES, dtype=torch.float16)
    for i, s in enumerate(seg):
        S[i] = torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt'))[:, :N_SAMPLES]
    return S.unfold(2, WIN, STRIDE).transpose(1, 2), np.array([s[3] for s in seg])

X_tr, age_tr = load_split('train')
X_va, age_va = load_split('val')
print(X_tr.shape, X_va.shape)

torch.Size([825, 112, 20, 512]) torch.Size([103, 112, 20, 512])


In [3]:
class CNN(nn.Module):
    def __init__(self, kernels, filters, pool):
        super().__init__()
        layers, f_in = [], 1
        for k, f in zip(kernels, filters):
            layers += [nn.Conv1d(C * f_in, C * f, k, groups=C), nn.ReLU()]
            f_in = f
        self.conv = nn.Sequential(*layers, nn.MaxPool1d(pool), nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.fc = nn.Linear(C * filters[-1], 1)

    def forward(self, x):
        return self.fc(self.conv(x)).squeeze(1)

In [4]:
def r2(y, p): return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

def metrics(log_pred, age):
    pred = np.exp(log_pred)
    return {'RMSE': np.sqrt(np.mean((pred - age) ** 2)), 'MAE': np.mean(np.abs(pred - age)),
            'R2 (age)': r2(age, pred), 'R2 (log age)': r2(np.log(age), log_pred),
            'RMSE (log)': np.sqrt(np.mean((log_pred - np.log(age)) ** 2))}

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return np.array([model(x.float().to(DEVICE)).mean().item() for x in X])

def train(cfg, name):
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = CNN(cfg['kernels'], cfg['filters'], cfg['pool']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=cfg['lr_factor'], patience=cfg['lr_patience'], min_lr=cfg['min_lr'])
    loss_fn = nn.MSELoss()
    y = torch.tensor(np.log(age_tr), dtype=torch.float32)
    n_win = X_tr.shape[1]
    child = torch.arange(len(X_tr)).repeat_interleave(n_win)
    win = torch.arange(n_win).repeat(len(X_tr))

    res = dict(name=name, config=cfg, params=sum(p.numel() for p in model.parameters()),
               history=[], best_epoch=None, age_train=age_tr, age_val=age_va)
    Path('runs').mkdir(exist_ok=True)
    best, wait = np.inf, 0
    for epoch in range(cfg['epochs']):
        t0 = time.time()
        model.train(); total = 0
        for b in torch.randperm(len(child)).split(cfg['batch']):
            c, w = child[b], win[b]
            loss = loss_fn(model(X_tr[c, w].float().to(DEVICE)), y[c].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.detach() * len(b)
        p_va, p_tr = predict(model, X_va), predict(model, X_tr)
        h = {'epoch': epoch + 1, 'lr': opt.param_groups[0]['lr'], 'train loss': total.item() / len(child),
             **{f'train {k}': float(v) for k, v in metrics(p_tr, age_tr).items()},
             **{f'val {k}': float(v) for k, v in metrics(p_va, age_va).items()},
             'seconds': time.time() - t0}
        res['history'].append(h)
        improved = h[cfg['select']] < best
        print(f"epoch {h['epoch']:3d}  lr {h['lr']:.1e}  train RMSE {h['train RMSE']:.3f}  val RMSE {h['val RMSE']:.3f}  "
              f"MAE {h['val MAE']:.3f}  RMSE (log) {h['val RMSE (log)']:.3f}  ({h['seconds']:.0f} s)" + ('  * best' if improved else ''))
        if improved:
            best, wait = h[cfg['select']], 0
            res.update(best_epoch=epoch + 1, state_dict={k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, pred_train=np.exp(p_tr), pred_val=np.exp(p_va))
        else:
            wait += 1
        sched.step(h[cfg['select']])
        torch.save(res, f'runs/{name}.pt')
        if wait >= cfg['patience']:
            break
    model.load_state_dict(res['state_dict'])
    return model, res

In [5]:
model, res = train(CFG, NAME)

epoch   1  lr 1.0e-03  train RMSE 1.312  val RMSE 1.160  MAE 0.805  RMSE (log) 0.568  (55 s)  * best


epoch   2  lr 1.0e-03  train RMSE 1.695  val RMSE 1.370  MAE 0.906  RMSE (log) 0.413  (53 s)


epoch   3  lr 1.0e-03  train RMSE 1.443  val RMSE 1.446  MAE 0.826  RMSE (log) 0.287  (52 s)


epoch   4  lr 1.0e-03  train RMSE 1.152  val RMSE 0.987  MAE 0.590  RMSE (log) 0.241  (52 s)  * best


epoch   5  lr 1.0e-03  train RMSE 1.079  val RMSE 1.082  MAE 0.666  RMSE (log) 0.313  (52 s)


epoch   6  lr 1.0e-03  train RMSE 1.190  val RMSE 1.157  MAE 0.700  RMSE (log) 0.276  (53 s)


... (26 outputs left out)

epoch  33  lr 3.1e-05  train RMSE 0.890  val RMSE 0.945  MAE 0.545  RMSE (log) 0.216  (56 s)


epoch  34  lr 3.1e-05  train RMSE 0.862  val RMSE 0.929  MAE 0.539  RMSE (log) 0.224  (57 s)


epoch  35  lr 3.1e-05  train RMSE 0.852  val RMSE 0.916  MAE 0.530  RMSE (log) 0.222  (57 s)


epoch  36  lr 1.6e-05  train RMSE 0.829  val RMSE 0.897  MAE 0.520  RMSE (log) 0.224  (57 s)


**RESULTS**:

In [6]:
best = res['history'][res['best_epoch'] - 1]
pd.DataFrame([{'best epoch': res['best_epoch'], **{k: best[k] for k in ['train RMSE', 'train MAE', 'val RMSE', 'val MAE']}}], index=[NAME]).round(3)

,best epoch,train RMSE,train MAE,val RMSE,val MAE
v4_k15_full,26,0.837,0.515,0.858,0.512


L